# **Insurance Claim Fraud Detection**

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    recall_score, roc_auc_score, f1_score,
    precision_recall_curve, roc_curve,fbeta_score,make_scorer
)
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from scipy.stats import loguniform
import joblib

In [ ]:
data = pd.read_csv('data\insurance_claims.csv')
data.head(5)

,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
0,328,48,521585,10/17/2014,OH,250/500,1000,1406.91,0,466132,...,YES,71610,6510,13020,52080,Saab,92x,2004,Y,NaN
1,228,42,342868,6/27/2006,IN,250/500,2000,1197.22,5000000,468176,...,?,5070,780,780,3510,Mercedes,E400,2007,Y,NaN
2,134,29,687698,9/6/2000,OH,100/300,2000,1413.14,5000000,430632,...,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,NaN
3,256,41,227811,5/25/1990,IL,250/500,2000,1415.74,6000000,608117,...,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y,NaN
4,228,44,367455,6/6/2014,IL,500/1000,1000,1583.91,6000000,610706,...,NO,6500,1300,650,4550,Accura,RSX,2009,N,NaN


In [55]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   months_as_customer           1000 non-null   int64  
 1   age                          1000 non-null   int64  
 2   policy_number                1000 non-null   int64  
 3   policy_bind_date             1000 non-null   object 
 4   policy_state                 1000 non-null   object 
 5   policy_csl                   1000 non-null   object 
 6   policy_deductable            1000 non-null   int64  
 7   policy_annual_premium        1000 non-null   float64
 8   umbrella_limit               1000 non-null   int64  
 9   insured_zip                  1000 non-null   int64  
 10  insured_sex                  1000 non-null   object 
 11  insured_education_level      1000 non-null   object 
 12  insured_occupation           1000 non-null   object 
 13  insured_hobbies    

In [56]:
print(f"  Dataset shape: {data.shape[0]} rows x {data.shape[1]} columns")
print(f"  Total features: {data.shape[1] - 1} (excluding target)")

  Dataset shape: 1000 rows x 40 columns
  Total features: 39 (excluding target)


In [57]:
data.isnull().sum()

months_as_customer                0
age                               0
policy_number                     0
policy_bind_date                  0
policy_state                      0
policy_csl                        0
policy_deductable                 0
policy_annual_premium             0
umbrella_limit                    0
insured_zip                       0
insured_sex                       0
insured_education_level           0
insured_occupation                0
insured_hobbies                   0
insured_relationship              0
capital-gains                     0
capital-loss                      0
incident_date                     0
incident_type                     0
collision_type                    0
incident_severity                 0
authorities_contacted            91
incident_state                    0
incident_city                     0
incident_location                 0
incident_hour_of_the_day          0
number_of_vehicles_involved       0
property_damage             

### Data Cleaning

In [58]:
# Replace '?' placeholders with 'Unknown' to preserve data integrity
data[['police_report_available', 'property_damage', 'collision_type']] = \
    data[['police_report_available', 'property_damage', 'collision_type']].replace('?', 'Unknown')

# Fill missing values in authorities_contacted
data['authorities_contacted'] = data['authorities_contacted'].fillna('Unknown')

# Convert date columns to datetime
data['policy_bind_date'] = pd.to_datetime(
    data['policy_bind_date'], errors='coerce', format='%m/%d/%Y'
)
data['incident_date'] = pd.to_datetime(
    data['incident_date'], errors='coerce', format='%m/%d/%Y'
)

# Drop Columns (do NOT contribute to fraud patterns)
data = data.drop(
    ['_c39', 'insured_zip', 'policy_number',
     'incident_location', 'auto_model', 'auto_make'],
    axis=1
)
print(f"  After cleaning: {data.shape[0]} rows x {data.shape[1]} columns")

  After cleaning: 1000 rows x 34 columns


### Feature Engineering

In [59]:
# Ratio of claim amount to annual premium — high ratio = suspicious
data['claim_ratio'] = data['total_claim_amount'] / data['policy_annual_premium']

# Extract year from incident date
data['incident_year'] = data['incident_date'].dt.year

# How old was the vehicle at time of incident? Older vehicles → easier to fake
data['vehicle_age'] = data['incident_year'] - data['auto_year']

# Days between policy start and incident — very short = suspicious
data['days_between_policy_incident'] = (
    data['incident_date'] - data['policy_bind_date']
).dt.days

# Parse policy CSL (e.g., "250/500") into numeric sub-features
csl_split = data['policy_csl'].str.split('/', expand=True).astype(int)
data['csl_per_person'] = csl_split[0]
data['csl_per_accident'] = csl_split[1]
data.drop('policy_csl', axis=1, inplace=True)

print(f"  New features: claim_ratio, incident_year, vehicle_age,")
print(f"    days_between_policy_incident, csl_per_person, csl_per_accident")
print(f"  Total features now: {data.shape[1]}")

  New features: claim_ratio, incident_year, vehicle_age,
    days_between_policy_incident, csl_per_person, csl_per_accident
  Total features now: 39


### Pre-Claim Model Features

In [60]:
y = data['fraud_reported'].str.strip().map({'Y': 1, 'N': 0})

X = data[[
    'months_as_customer', 'age', 'policy_bind_date', 'policy_state',
    'policy_deductable', 'policy_annual_premium', 'insured_sex',
    'insured_education_level', 'insured_occupation','insured_hobbies',
    'insured_relationship', 'incident_date', 'incident_type',
    'collision_type', 'incident_severity', 'authorities_contacted',
    'incident_state','incident_city', 'incident_hour_of_the_day',
    'number_of_vehicles_involved', 'property_damage', 'bodily_injuries',
    'witnesses', 'police_report_available', 'auto_year',
    'total_claim_amount', 'claim_ratio', 'incident_year',
    'vehicle_age', 'csl_per_person', 'csl_per_accident',
    'days_between_policy_incident'
]].copy()

print(f"  Feature matrix shape: {X.shape}")
print(f"  Target distribution:\n{y.value_counts(normalize=True).to_string()}")

  Feature matrix shape: (1000, 32)
  Target distribution:
fraud_reported
0    0.753
1    0.247


I manually selected features for the model. I excluded injury_claim, property_claim, and vehicle_claim because they are components of total_claim_amount. Including them would introduce multicollinearity (redundancy) and confuse the model

In [61]:
X.head()

,months_as_customer,age,policy_bind_date,policy_state,policy_deductable,policy_annual_premium,insured_sex,insured_education_level,insured_occupation,insured_hobbies,...,witnesses,police_report_available,auto_year,total_claim_amount,claim_ratio,incident_year,vehicle_age,csl_per_person,csl_per_accident,days_between_policy_incident
0,328,48,2014-10-17,OH,1000,1406.91,MALE,MD,craft-repair,sleeping,...,2,YES,2004,71610,50.898778,2015,11,250,500,100
1,228,42,2006-06-27,IN,2000,1197.22,MALE,MD,machine-op-inspct,reading,...,0,Unknown,2007,5070,4.234811,2015,8,250,500,3130
2,134,29,2000-09-06,OH,2000,1413.14,FEMALE,PhD,sales,board-games,...,3,NO,2007,34650,24.519864,2015,8,100,300,5282
3,256,41,1990-05-25,IL,2000,1415.74,FEMALE,PhD,armed-forces,board-games,...,2,NO,2014,63400,44.782234,2015,1,250,500,8996
4,228,44,2014-06-06,IL,1000,1583.91,MALE,Associate,sales,board-games,...,1,NO,2009,6500,4.103769,2015,6,500,1000,256


### Train-Test Split

In [62]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"  Train set: {X_train.shape[0]} samples (fraud rate: {y_train.mean():.2%})")
print(f"  Test set:  {X_test.shape[0]} samples (fraud rate: {y_test.mean():.2%})")


  Train set: 800 samples (fraud rate: 24.75%)
  Test set:  200 samples (fraud rate: 24.50%)


In [63]:
numeric_features = [
    'months_as_customer', 'age', 'policy_deductable', 'policy_annual_premium',
    'incident_hour_of_the_day', 'number_of_vehicles_involved', 'bodily_injuries',
    'witnesses', 'auto_year', 'total_claim_amount', 'claim_ratio',
    'incident_year', 'vehicle_age', 'csl_per_person', 'csl_per_accident',
    'days_between_policy_incident'
]

categorical_features = [
    'policy_state', 'insured_sex', 'insured_education_level',
    'insured_occupation', 'insured_relationship','insured_hobbies',
    'incident_type', 'collision_type', 'incident_severity',
    'authorities_contacted', 'incident_state','incident_city',
    'property_damage', 'police_report_available'
]

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

print(f"  Numeric features:   {len(numeric_features)}")
print(f"  Categorical features: {len(categorical_features)}")

  Numeric features:   16
  Categorical features: 14


### Model building (with Hyperparameter Optimization - Random Search)

In [64]:
# ── Logistic Regression ────────────────────────────────────
print("\n  Logistic Regression...")
lr_pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(random_state=42, max_iter=2000))
])

lr_param_dist = {
    'model__penalty': ['l1', 'l2'],
    'model__C': loguniform(1e-4, 10),
    'model__solver': ['liblinear', 'saga']
}

f2_scorer = make_scorer(fbeta_score, beta=2)
lr_search = RandomizedSearchCV(
    lr_pipeline, lr_param_dist,
    n_iter=50, cv=5, n_jobs=-1,
    random_state=42, verbose=0,
    scoring=f2_scorer
)
lr_search.fit(X_train, y_train)
print(f"    Best CV F2 Score: {lr_search.best_score_:.4f}")
print(f"    Best params: {lr_search.best_params_}")


  Logistic Regression...
    Best CV F2 Score: 0.8326
    Best params: {'model__C': np.float64(0.09163741808778776), 'model__penalty': 'l1', 'model__solver': 'liblinear'}


In [65]:
# ── Random Forest ──────────────────────────────────────────
print("\n  Random Forest...")
rf_pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_param_dist = {
    'model__n_estimators': [int(x) for x in np.linspace(start=100, stop=1000, num=10)],
    'model__max_depth': [int(x) for x in np.linspace(start=10, stop=110, num=11)],
    'model__min_samples_leaf': [1, 2, 4, 10, 20, 50, 100],
    'model__min_samples_split': [2, 3, 4, 5, 8, 10, 20, 50, 100, 200],
    'model__max_features': ['sqrt', 'log2', 0.3, 0.5]
}

rf_search = RandomizedSearchCV(
    rf_pipeline, rf_param_dist,
    cv=5, n_iter=50, n_jobs=-1,
    random_state=42, verbose=0,
    scoring=f2_scorer
)
rf_search.fit(X_train, y_train)
print(f"Best CV F2 Score: {rf_search.best_score_:.4f}")
print(f"Best params: {rf_search.best_params_}")


  Random Forest...
Best CV F2 Score: 0.8068
Best params: {'model__n_estimators': 300, 'model__min_samples_split': 20, 'model__min_samples_leaf': 4, 'model__max_features': 0.5, 'model__max_depth': 80}


In [66]:
print("\n  XGBoost...")
xgb_pipeline = ImbPipeline(steps=[
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

xgb_param_dist = {
    'model__n_estimators': [int(x) for x in np.linspace(start=50, stop=500, num=10)],
    'model__max_depth': [int(x) for x in np.linspace(start=3, stop=10, num=7)],
    'model__learning_rate': loguniform(0.1, 0.3)
}

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    xgb_param_dist,
    n_iter=50,
    cv=5,
    random_state=42,
    verbose=0,
    scoring=f2_scorer
)

xgb_search.fit(X_train, y_train)

print(f"Best CV F2 Score: {xgb_search.best_score_:.4f}")
print(f"Best params: {xgb_search.best_params_}")


  XGBoost...
Best CV F2 Score: 0.7846
Best params: {'model__learning_rate': np.float64(0.10077933409688876), 'model__max_depth': 3, 'model__n_estimators': 50}


### Re-training XGBoost with scale_pos_weight

In [67]:
from scipy.stats import loguniform
import numpy as np

print("\n  XGBoost (scale_pos_weight + RandomizedSearchCV)...")

# Same ratio as before
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight_ratio = neg_count / pos_count

# Pipeline WITHOUT SMOTE, using scale_pos_weight instead
xgb_balanced_search_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', XGBClassifier(
        scale_pos_weight=scale_pos_weight_ratio,
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

xgb_balanced_param_dist = {
    'model__n_estimators': [int(x) for x in np.linspace(50, 500, 10)],
    'model__max_depth': [int(x) for x in np.linspace(3, 10, 7)],
    'model__learning_rate': loguniform(0.01, 0.3),
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 3, 5, 7]
}

xgb_balanced_search = RandomizedSearchCV(
    xgb_balanced_search_pipeline,
    xgb_balanced_param_dist,
    n_iter=50,
    cv=5,
    random_state=42,
    verbose=1,
    scoring=f2_scorer
)

xgb_balanced_search.fit(X_train, y_train)

print(f"\n  Best CV F2 Score: {xgb_balanced_search.best_score_:.4f}")
print(f"  Best params: {xgb_balanced_search.best_params_}")

# Evaluate on test set
y_pred_xgb_bal_tuned = xgb_balanced_search.predict(X_test)
y_proba_xgb_bal_tuned = xgb_balanced_search.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred_xgb_bal_tuned)
rec  = recall_score(y_test, y_pred_xgb_bal_tuned)
f1   = f1_score(y_test, y_pred_xgb_bal_tuned)
f2   = fbeta_score(y_test, y_pred_xgb_bal_tuned, beta=2)
roc  = roc_auc_score(y_test, y_proba_xgb_bal_tuned)

print("\n  ╔══ XGBoost (Balanced + Tuned) ══════════════════╗")
print(f"  ║  Accuracy:  {acc:.4f}")
print(f"  ║  Recall:    {rec:.4f}")
print(f"  ║  F1 Score:  {f1:.4f}")
print(f"  ║  F2 Score:  {f2:.4f}")
print(f"  ║  ROC-AUC:   {roc:.4f}")
print("  ╚════════════════════════════════════════════════╝")

print("\n  Classification Report:")
print(classification_report(y_test, y_pred_xgb_bal_tuned,
                            target_names=['Legitimate', 'Fraud']))


  XGBoost (scale_pos_weight + RandomizedSearchCV)...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

  Best CV F2 Score: 0.8326
  Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': np.float64(0.015019490572374374), 'model__max_depth': 8, 'model__min_child_weight': 5, 'model__n_estimators': 50, 'model__subsample': 1.0}

  ╔══ XGBoost (Balanced + Tuned) ══════════════════╗
  ║  Accuracy:  0.8450
  ║  Recall:    0.8571
  ║  F1 Score:  0.7304
  ║  F2 Score:  0.8015
  ║  ROC-AUC:   0.8430
  ╚════════════════════════════════════════════════╝

  Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.95      0.84      0.89       151
       Fraud       0.64      0.86      0.73        49

    accuracy                           0.84       200
   macro avg       0.79      0.85      0.81       200
weighted avg       0.87      0.84      0.85       200



In [69]:
models = {
    'Logistic Regression': lr_search.best_estimator_,
    'Random Forest': rf_search.best_estimator_,
    'XGBoost': xgb_balanced_search.best_estimator_
}

results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    f2 = fbeta_score(y_test, y_pred, beta=2)

    results[name] = {
        'Accuracy': acc,
        'Recall': rec,
        'F1 Score': f1,
        'F2 Score': f2,     
        'ROC-AUC': roc,
        'Predictions': y_pred,
        'Probabilities': y_proba
    }

    print(f"\n  ╔══ {name} ════════════════════════════════╗")
    print(f"  ║  Accuracy:  {acc:.4f}")
    print(f"  ║  Recall:    {rec:.4f}")
    print(f"  ║  F1 Score:  {f1:.4f}")
    print(f"  ║  F2 Score:  {f2:.4f}")  
    print(f"  ║  ROC-AUC:   {roc:.4f}")
    print(f"  ╚════════════════════════════════════════╝")


    print(f"\n  Classification Report:")
    report = classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'])
    for line in report.split('\\n'):
        print(f"    {line}")


  ╔══ Logistic Regression ════════════════════════════════╗
  ║  Accuracy:  0.8450
  ║  Recall:    0.8571
  ║  F1 Score:  0.7304
  ║  F2 Score:  0.8015
  ║  ROC-AUC:   0.8490
  ╚════════════════════════════════════════╝

  Classification Report:
                  precision    recall  f1-score   support

  Legitimate       0.95      0.84      0.89       151
       Fraud       0.64      0.86      0.73        49

    accuracy                           0.84       200
   macro avg       0.79      0.85      0.81       200
weighted avg       0.87      0.84      0.85       200


  ╔══ Random Forest ════════════════════════════════╗
  ║  Accuracy:  0.8250
  ║  Recall:    0.7143
  ║  F1 Score:  0.6667
  ║  F2 Score:  0.6944
  ║  ROC-AUC:   0.8340
  ╚════════════════════════════════════════╝

  Classification Report:
                  precision    recall  f1-score   support

  Legitimate       0.90      0.86      0.88       151
       Fraud       0.62      0.71      0.67        49

    accuracy 

In [18]:
# import joblib

# # saving the model
# joblib.dump(lr_search.best_estimator_, 'best_model.pkl')

### Confusion Matrix Heatmaps - 

In [ ]:
# ── Confusion Matrix Heatmaps ──────────────────────────
print("  Generating confusion matrix heatmaps...")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, name in enumerate(models.keys()):
    cm = confusion_matrix(y_test, results[name]['Predictions'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
        xticklabels=['Legitimate', 'Fraud'],
        yticklabels=['Legitimate', 'Fraud'],
        annot_kws={'size': 14}
    )
    axes[idx].set_title(f'{name}\nAccuracy: {results[name]["Accuracy"]:.3f}',
                         fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Actual', fontsize=11)
    axes[idx].set_xlabel('Predicted', fontsize=11)
plt.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: plots/confusion_matrices.png")

  Generating confusion matrix heatmaps...
Saved: plots/confusion_matrices.png


### Model Comparison Chart

In [ ]:
# ── Model Comparison Chart ──────────────────────────
print("  Generating model comparison chart...")
metrics_list = ['Accuracy', 'Recall', 'F1 Score', 'F2 Score', 'ROC-AUC']
metric_keys = ['Accuracy', 'Recall', 'F1 Score', 'F2 Score', 'ROC-AUC']
x = np.arange(len(metrics_list))
width = 0.25
model_names = list(models.keys())
colors = ['#2ecc71', '#3498db', '#e74c3c']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (name, color) in enumerate(zip(model_names, colors)):
    values = [results[name][key] for key in metric_keys]
    bars = ax.bar(x + i * width, values, width, label=name, color=color, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold'
        )

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_list, fontsize=11)
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/model_comparison.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: plots/model_comparison.png")

  Generating model comparison chart...
Saved: plots/model_comparison.png


### ROC Curve

In [ ]:
print("  ▸ Generating ROC curve...")
plt.figure(figsize=(8, 6))
for name in models.keys():
    fpr, tpr, _ = roc_curve(y_test, results[name]['Probabilities'])
    auc = results[name]['ROC-AUC']
    plt.plot(fpr, tpr, linewidth=2.5, label=f'{name} (AUC = {auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curve.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: plots/roc_curve.png")

  ▸ Generating ROC curve...
Saved: plots/roc_curve.png


## Model Explainability with SHAP

In [77]:
# ── SHAP Explainability ────────────────────────────────────────────────
import shap

# Get the best Logistic Regression pipeline and extract preprocessor & model
best_lr = lr_search.best_estimator_
preprocessor = best_lr.named_steps['preprocess']
lr_model = best_lr.named_steps['model']

# Preprocess the training and test sets
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# Clean the feature names for clean plotting
clean_feature_names = [
    name.replace("num__", "").replace("cat__", "").replace("_", " ").title()
    for name in preprocessor.get_feature_names_out()
]

# Convert preprocessed arrays to pandas DataFrames with clean column names
X_train_trans_df = pd.DataFrame(X_train_trans, columns=clean_feature_names)
X_test_trans_df = pd.DataFrame(X_test_trans, columns=clean_feature_names)

# Initialize shap.LinearExplainer using the training background data
explainer = shap.Explainer(lr_model, X_train_trans_df)
shap_values = explainer(X_test_trans_df)

# Ensure the assets/ folder exists
os.makedirs('assets', exist_ok=True)

# Generate and save the bar plot of mean absolute SHAP values (top 15 features)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_trans_df, plot_type="bar", max_display=15, show=False)
plt.title("Mean Absolute SHAP Values (Top 15 Features)", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(r'C:\Users\Paras\Desktop\Projects\Insurance_Claim_Fraud_Detection\plots\shap_bar.png', bbox_inches='tight', dpi=150)
plt.close()
print("Saved: plots/shap_bar.png")

Background dataset has 800 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=800 when initializing the masker.


Saved: plots/shap_bar.png


### Understanding the SHAP Plots

SHAP (SHapley Additive exPlanations) values break down the contribution of each feature to the model's final prediction score.

**Bar Plot of Mean Absolute SHAP Values (`plotss/shap_bar.png`)**:
   - **What it shows**: The bar plot displays the average magnitude of impact each feature has on the prediction, without considering direction.
   - **Interpretation**: It answers the question: "On average, how much does this feature shift the fraud probability score?" It is a direct measure of global feature importance. Longer bars indicate features that the model relies on most heavily.